# Neural Network: Flexible Nonlinear Model

This notebook implements a simple neural network to test whether flexible nonlinear models further improve predictive performance.

**Goal**: Explore highly flexible models that can learn complex patterns.

## Model Architecture:
- **Input Layer**: 17 features
- **Hidden Layers**: Fully connected layers with ReLU activation
- **Output Layer**: Single neuron with sigmoid activation (binary classification)
- **Regularization**: Dropout and early stopping

## Evaluation Metrics:
- **F1-Score** (primary): Mean ± standard deviation across 5 folds
- **ROC-AUC**, **Precision**, **Recall**

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

# Import the evaluate_model function
from evaluate_model import evaluate_model

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load preprocessed training data
print("Loading preprocessed training data...")
train_df = pd.read_csv('../data/train_data_preprocessed.csv')
print(f"Training data shape: {train_df.shape}")

# Prepare features and target
X = train_df.drop('Revenue', axis=1)
y = train_df['Revenue']

print(f"\nFeatures shape: {X.shape}")
print(f"Features: {list(X.columns)}")

In [ ]:
# Setup cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2. Feature Standardization

Neural networks require standardized features. We'll use a pipeline to prevent leakage.

In [ ]:
from sklearn.pipeline import Pipeline

print("Feature standardization will be applied within CV folds to prevent data leakage.")

## 3. Simple Neural Network

Start with a simple 2-layer architecture.

In [ ]:
# Create pipeline with standardization and neural network
nn_simple = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,  # L2 regularization
        batch_size=32,
        learning_rate='adaptive',
        max_iter=200,
        early_stopping=True,
        validation_fraction=0.2,
        random_state=42
    ))
])

print("Training Simple Neural Network (2 hidden layers: 64, 32)...")
results_nn_simple = evaluate_model(nn_simple, X, y, cv, "Neural Network (Simple)")

## 4. Hyperparameter Tuning

Tune key hyperparameters:
- **hidden_layer_sizes**: Network architecture
- **alpha**: L2 regularization strength
- **learning_rate_init**: Initial learning rate

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Create pipeline
nn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', MLPClassifier(
        activation='relu',
        solver='adam',
        batch_size=32,
        learning_rate='adaptive',
        max_iter=200,
        early_stopping=True,
        validation_fraction=0.2,
        random_state=42
    ))
])

# Hyperparameter grid
param_distributions = {
    'classifier__hidden_layer_sizes': [
        (32,), (64,), (128,),
        (64, 32), (128, 64), (128, 64, 32),
        (256, 128), (256, 128, 64)
    ],
    'classifier__alpha': [0.0001, 0.001, 0.01, 0.1],
    'classifier__learning_rate_init': [0.001, 0.01, 0.1]
}

# Randomized search
random_search_nn = RandomizedSearchCV(
    nn_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

print("Training Neural Network with Randomized Search...")
print("This may take several minutes...")
random_search_nn.fit(X, y)

print(f"\nBest parameters: {random_search_nn.best_params_}")
print(f"Best F1-Score: {random_search_nn.best_score_:.4f}")

In [ ]:
# Evaluate best neural network
results_nn_tuned = evaluate_model(random_search_nn.best_estimator_, X, y, cv, "Neural Network (Tuned)")

## 5. Model Comparison

In [ ]:
print(f"\n{'='*60}")
print("NEURAL NETWORK COMPARISON")
print(f"{'='*60}")
print(f"\n{'Metric':<15} {'Simple NN':<15} {'Tuned NN':<15}")
print(f"{'-'*45}")
print(f"{'F1-Score':<15} {results_nn_simple['cv_f1']:<15.4f} {results_nn_tuned['cv_f1']:<15.4f}")
print(f"{'Precision':<15} {results_nn_simple['cv_precision']:<15.4f} {results_nn_tuned['cv_precision']:<15.4f}")
print(f"{'Recall':<15} {results_nn_simple['cv_recall']:<15.4f} {results_nn_tuned['cv_recall']:<15.4f}")
print(f"{'ROC-AUC':<15} {results_nn_simple['cv_roc_auc']:<15.4f} {results_nn_tuned['cv_roc_auc']:<15.4f}")

print(f"\n{'='*60}")
print("KEY INSIGHTS")
print(f"{'='*60}")
print(f"\nBest Architecture: {random_search_nn.best_params_['classifier__hidden_layer_sizes']}")
print(f"Best Alpha (L2): {random_search_nn.best_params_['classifier__alpha']}")
print(f"Best Learning Rate: {random_search_nn.best_params_['classifier__learning_rate_init']}")

print(f"\nBias-Variance Tradeoff:")
print(f"  - Deeper networks: Lower bias, higher variance")
print(f"  - L2 regularization (alpha): Controls variance")
print(f"  - Early stopping: Prevents overfitting")
print(f"  - Dropout (implicit in MLPClassifier): Reduces variance")

## 6. Learning Curves (Optional)

In [ ]:
# Train a single model to examine learning curves
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train with loss curve tracking
nn_curve = MLPClassifier(
    **{k.replace('classifier__', ''): v for k, v in random_search_nn.best_params_.items()},
    activation='relu',
    solver='adam',
    batch_size=32,
    learning_rate='adaptive',
    max_iter=200,
    random_state=42,
    verbose=False
)

nn_curve.fit(X_train_scaled, y_train)

# Plot loss curve
plt.figure(figsize=(10, 6))
plt.plot(nn_curve.loss_curve_)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Neural Network Training Loss')
plt.grid(True)
plt.show()

## 7. Summary

In [ ]:
print(f"\n{'='*60}")
print("NEURAL NETWORK SUMMARY")
print(f"{'='*60}")
print(f"\nStrengths:")
print(f"  - Highly flexible: can learn complex nonlinear patterns")
print(f"  - Automatic feature learning through hidden layers")
print(f"  - Regularization through L2 penalty and early stopping")

print(f"\nWeaknesses:")
print(f"  - Requires careful tuning of many hyperparameters")
print(f"  - Sensitive to feature scaling")
print(f"  - Can be prone to overfitting on small datasets")
print(f"  - Less interpretable than tree-based models")

print(f"\nComparison with Other Models:")
print(f"  - Compare with XGBoost, Random Forest, and Regularized Logistic Regression")
print(f"  - Does added flexibility improve F1-Score and ROC-AUC?")
print(f"  - Consider computational cost vs. performance gain")

## 8. Final Model Selection

After training all models:
1. **Baseline**: Logistic Regression (no regularization)
2. **Regularized Linear**: L1/L2 Logistic Regression
3. **Tree-Based**: Decision Tree, Random Forest
4. **Boosting**: XGBoost
5. **Neural Network**: MLPClassifier

**Next Steps:**
- Create a comparison table of all models
- Select the best model based on F1-Score and ROC-AUC
- Evaluate the selected model on the holdout set (Part 2)